# 🦠 COVID-19 Global Data Analysis
**Live data · disease.sh API** | Advanced EDA with Plotly

---

In [ ]:
import pandas as pd, numpy as np, requests
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
print("Libraries loaded")

## 1. Load Live Country Data

In [ ]:
r = requests.get("https://disease.sh/v3/covid-19/countries?sort=cases", timeout=20)
df = pd.json_normalize(r.json())
df = df.rename(columns={
    "country":"Country","cases":"Total Cases","deaths":"Total Deaths",
    "recovered":"Recovered","active":"Active","todayCases":"Today Cases",
    "todayDeaths":"Today Deaths","casesPerOneMillion":"Cases/1M",
    "deathsPerOneMillion":"Deaths/1M","population":"Population",
    "continent":"Continent","countryInfo.lat":"Lat","countryInfo.long":"Lon",
})
df["Mortality Rate (%)"] = (df["Total Deaths"] / df["Total Cases"] * 100).round(2)
df["Recovery Rate (%)"]  = (df["Recovered"]    / df["Total Cases"] * 100).round(2)
print(f"Loaded {len(df)} countries")
df[["Country","Continent","Total Cases","Total Deaths","Mortality Rate (%)"]].head(10)

## 2. Global Summary

In [ ]:
g = requests.get("https://disease.sh/v3/covid-19/all", timeout=15).json()
pd.DataFrame([{
    "Total Cases":g["cases"],"Total Deaths":g["deaths"],
    "Recovered":g["recovered"],"Active":g["active"],
    "Countries":g["affectedCountries"]
}])

## 3. Continent Aggregation

In [ ]:
cont = df.groupby("Continent").agg(
    Countries=("Country","count"),
    Total_Cases=("Total Cases","sum"),
    Total_Deaths=("Total Deaths","sum"),
    Avg_Mortality=("Mortality Rate (%)","mean"),
).reset_index().sort_values("Total_Cases", ascending=False)
cont["Avg_Mortality"] = cont["Avg_Mortality"].round(2)
cont

## 4. World Choropleth Map

In [ ]:
fig = px.choropleth(df, locations="Country", locationmode="country names",
    color="Total Cases", hover_name="Country",
    hover_data=["Total Deaths","Mortality Rate (%)","Population"],
    color_continuous_scale="Reds", title="COVID-19 — Total Cases by Country")
fig.update_layout(height=500)
fig.show()

## 5. Top 20 Countries

In [ ]:
top20 = df.nlargest(20, "Total Cases")
fig = px.bar(top20, x="Total Cases", y="Country", orientation="h",
             color="Mortality Rate (%)", color_continuous_scale="RdYlGn_r",
             title="Top 20 Countries — Cases colored by Mortality Rate")
fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
fig.show()

## 6. Continent Pie Charts

In [ ]:
fig = make_subplots(rows=1, cols=2, specs=[[{"type":"pie"},{"type":"pie"}]])
fig.add_trace(go.Pie(labels=cont["Continent"], values=cont["Total_Cases"],
                     name="Cases", hole=0.4), row=1, col=1)
fig.add_trace(go.Pie(labels=cont["Continent"], values=cont["Total_Deaths"],
                     name="Deaths", hole=0.4), row=1, col=2)
fig.update_layout(title="Cases vs Deaths Share by Continent", height=420)
fig.show()

## 7. Scatter: Cases/1M vs Deaths/1M

In [ ]:
fig = px.scatter(
    df.dropna(subset=["Cases/1M","Deaths/1M","Continent"]),
    x="Cases/1M", y="Deaths/1M", color="Continent",
    size="Population", hover_name="Country", trendline="ols",
    log_x=True, log_y=True,
    title="Cases per Million vs Deaths per Million (log scale)")
fig.update_layout(height=500)
fig.show()

## 8. Historical 90-Day Global Trend

In [ ]:
hist = requests.get(
    "https://disease.sh/v3/covid-19/historical/all?lastdays=90",
    timeout=20).json()
h = pd.DataFrame({
    "Date":  list(hist["cases"].keys()),
    "Cases": list(hist["cases"].values()),
    "Deaths":list(hist["deaths"].values()),
})
h["Date"] = pd.to_datetime(h["Date"])
h["Daily Cases"]  = h["Cases"].diff().clip(lower=0)
h["Daily Deaths"] = h["Deaths"].diff().clip(lower=0)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["Cumulative Cases & Deaths","Daily New Cases"])
fig.add_trace(go.Scatter(x=h["Date"], y=h["Cases"],
    name="Cases", line=dict(color="steelblue")), row=1, col=1)
fig.add_trace(go.Scatter(x=h["Date"], y=h["Deaths"],
    name="Deaths", line=dict(color="crimson")), row=1, col=1)
fig.add_trace(go.Bar(x=h["Date"], y=h["Daily Cases"],
    name="Daily Cases", marker_color="steelblue"), row=2, col=1)
fig.update_layout(height=600, title="Global COVID-19 Last 90 Days",
                  hovermode="x unified")
fig.show()

## 9. Correlation Matrix

In [ ]:
num_cols = ["Total Cases","Total Deaths","Recovered","Active",
           "Cases/1M","Deaths/1M","Mortality Rate (%)","Recovery Rate (%)"]
corr = df[num_cols].corr()
fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                text_auto=".2f", title="Correlation Matrix", aspect="auto")
fig.update_layout(height=500)
fig.show()

## 10. Key Insights

In [ ]:
print("=== COVID-19 KEY INSIGHTS ===")
print(f"Countries tracked : {len(df)}")
print(f"Top 3 by cases    : {df.nlargest(3,'Total Cases')['Country'].tolist()}")
top_mort = df[df['Total Cases']>10000].nlargest(3,'Mortality Rate (%)')
print("\nHighest mortality (>10K cases):")
for _, row in top_mort.iterrows():
    print(f"  {row['Country']}: {row['Mortality Rate (%)']:.2f}%")
avg_m = df[df['Total Cases']>10000]['Mortality Rate (%)'].mean()
avg_r = df['Recovery Rate (%)'].mean()
print(f"\nAvg mortality (>10K cases): {avg_m:.2f}%")
print(f"Avg recovery rate          : {avg_r:.2f}%")